In [1]:
# Some structure and comments in this notebook were written with help from ChatGPT.
# I understand the code and made sure it matches the assignment requirements.
# Also, I had done pip install spacy but deleted it after it was done. 

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

In [2]:
train_file = "A06_train.csv"
test_file = "A06_test.csv"

train_df = pd.read_csv(train_file, header=None, names=["sentiment", "text"])
test_df = pd.read_csv(test_file, header=None, names=["sentiment", "text"])

train_df["text"] = train_df["text"].astype(str)
test_df["text"] = test_df["text"].astype(str)

X_train = train_df["text"]
y_train = train_df["sentiment"]

X_test = test_df["text"]
y_test = test_df["sentiment"]

print("Training data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

print("\nTraining label counts:")
print(train_df["sentiment"].value_counts())

print("\nTest label counts:")
print(test_df["sentiment"].value_counts())

Training data shape: (26499, 2)
Test data shape: (6625, 2)

Training label counts:
sentiment
 0    11510
 1     9621
-1     5368
Name: count, dtype: int64

Test label counts:
sentiment
 0    2877
 1    2429
-1    1319
Name: count, dtype: int64


In [3]:
results = []

def test_model(model_name, vectorizer):
    model = Pipeline([
        ("vectorizer", vectorizer),
        ("classifier", LinearSVC(random_state=42, max_iter=5000))
    ])
    
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, predictions)
    
    results.append({
        "Model": model_name,
        "Accuracy": accuracy
    })
    
    print(model_name)
    print("Accuracy:", round(accuracy, 4))
    print()
    
    return model, predictions

In [4]:
baseline_vectorizer = CountVectorizer()

baseline_model, baseline_predictions = test_model(
    "Baseline: Bag-of-Words",
    baseline_vectorizer
)

Baseline: Bag-of-Words
Accuracy: 0.6325



In [5]:
stopword_vectorizer = CountVectorizer(stop_words="english")

stopword_model, stopword_predictions = test_model(
    "Enhanced 1: Stopword Removal",
    stopword_vectorizer
)

Enhanced 1: Stopword Removal
Accuracy: 0.6195



In [6]:
bigram_vectorizer = CountVectorizer(ngram_range=(1, 2))

bigram_model, bigram_predictions = test_model(
    "Enhanced 2: Unigrams + Bigrams",
    bigram_vectorizer
)

Enhanced 2: Unigrams + Bigrams
Accuracy: 0.6503



In [7]:
combined_vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

combined_model, combined_predictions = test_model(
    "Enhanced 3: Stopwords + Bigrams",
    combined_vectorizer
)

Enhanced 3: Stopwords + Bigrams
Accuracy: 0.6533



In [8]:
!python -m spacy download en_core_web_md

     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.3/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.3/33.5 MB ? eta 


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import spacy

try:
    nlp = spacy.load("en_core_web_md")
    print("Loaded spaCy model successfully.")
except OSError:
    print("spaCy model not found.")
    print("Run the next cell once, then run this cell again.")

Loaded spaCy model successfully.


In [10]:
def texts_to_vectors(texts):
    vectors = []
    
    for doc in nlp.pipe(texts, batch_size=100):
        vectors.append(doc.vector)
    
    return np.array(vectors)

X_train_vectors = texts_to_vectors(X_train)
X_test_vectors = texts_to_vectors(X_test)

print("Training vector shape:", X_train_vectors.shape)
print("Test vector shape:", X_test_vectors.shape)

Training vector shape: (26499, 300)
Test vector shape: (6625, 300)


In [11]:
embedding_classifier = LinearSVC(random_state=42, max_iter=5000)

embedding_classifier.fit(X_train_vectors, y_train)
embedding_predictions = embedding_classifier.predict(X_test_vectors)

embedding_accuracy = accuracy_score(y_test, embedding_predictions)

results.append({
    "Model": "Enhanced 4: spaCy Embeddings",
    "Accuracy": embedding_accuracy
})

print("Enhanced 4: spaCy Embeddings")
print("Accuracy:", round(embedding_accuracy, 4))

Enhanced 4: spaCy Embeddings
Accuracy: 0.5807


In [12]:
results_df = pd.DataFrame(results)

baseline_accuracy = results_df.loc[
    results_df["Model"] == "Baseline: Bag-of-Words",
    "Accuracy"
].iloc[0]

results_df["Change from Baseline"] = results_df["Accuracy"] - baseline_accuracy

results_df

,Model,Accuracy,Change from Baseline
0,Baseline: Bag-of-Words,0.632453,0.000000
1,Enhanced 1: Stopword Removal,0.619472,-0.012981
2,Enhanced 2: Unigrams + Bigrams,0.650264,0.017811
3,Enhanced 3: Stopwords + Bigrams,0.653283,0.020830
4,Enhanced 4: spaCy Embeddings,0.580679,-0.051774


In [13]:
best_model = results_df.sort_values(by="Accuracy", ascending=False).iloc[0]

print("Best model:", best_model["Model"])
print("Best accuracy:", round(best_model["Accuracy"], 4))
print("Change from baseline:", round(best_model["Change from Baseline"], 4))

print("Classification report for Stopwords + Bigrams model:")
print(classification_report(y_test, combined_predictions))

Best model: Enhanced 3: Stopwords + Bigrams
Best accuracy: 0.6533
Change from baseline: 0.0208
Classification report for Stopwords + Bigrams model:
              precision    recall  f1-score   support

          -1       0.64      0.49      0.56      1319
           0       0.62      0.71      0.66      2877
           1       0.70      0.67      0.69      2429

    accuracy                           0.65      6625
   macro avg       0.65      0.63      0.64      6625
weighted avg       0.66      0.65      0.65      6625



LLM Use Reflection

I used ChatGPT to help structure the notebook code and write the analysis explanations. 
My prompts asked for the code structure, formatting, code related to spacy because I have not used it before, and information about the other libraries needed and any specific functions from them. This was helpful because it gave me a clean starting structure, but I still need to understand and explain the code myself. I learned how to compare bag-of-words, enhanced CountVectorizer features, and spaCy embeddings using the same classifier.